# Workbook v5: 1 BIP Close Fee Analysis

## Parameters
- **Open Fee**: 0%
- **Close Fee**: max(0.01% notional, 20% profit)
- **Liquidation Threshold**: 70% margin lost
- **Leverage**: 500x - 1000x

## Scenarios
1. **Perfect Oracle** - No latency, no front-running possible
2. **Front-runnable Oracle** - 400ms Pyth latency, arbitrage possible

In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Cell 1: Asset Parameters & Fee Structure

In [2]:
@dataclass
class Asset:
    name: str
    annual_vol: float  # Annual volatility
    hourly_vol: float  # Derived from annual
    second_vol: float  # Per-second volatility for high-freq
    
    @classmethod
    def create(cls, name: str, annual_vol: float):
        hourly_vol = annual_vol / np.sqrt(365 * 24)
        second_vol = annual_vol / np.sqrt(365 * 24 * 3600)
        return cls(name, annual_vol, hourly_vol, second_vol)

# Asset definitions
ASSETS = {
    'BTC': Asset.create('BTC', 0.60),   # 60% annual vol
    'ETH': Asset.create('ETH', 0.75),   # 75% annual vol
    'SOL': Asset.create('SOL', 1.00),   # 100% annual vol
}

# Fee structure v5
@dataclass
class FeeStructure:
    open_fee_rate: float = 0.0         # 0%
    close_fee_min: float = 0.0001      # 0.01% = 1 BIP
    profit_fee_rate: float = 0.20      # 20% of profit
    liquidation_threshold: float = 0.70  # 70% margin lost (standard)
    funding_rate_hourly: float = 0.0001  # 0.01% per hour

FEES = FeeStructure()

print("=== FEE STRUCTURE v5 ===")
print(f"Open Fee: {FEES.open_fee_rate*100:.2f}%")
print(f"Close Fee: max({FEES.close_fee_min*100:.2f}% notional, {FEES.profit_fee_rate*100:.0f}% profit)")
print(f"Liquidation: {FEES.liquidation_threshold*100:.0f}% margin lost")
print(f"Funding: {FEES.funding_rate_hourly*100:.3f}% per hour")
print()
print("=== ASSETS ===")
for name, asset in ASSETS.items():
    print(f"{name}: Annual Vol={asset.annual_vol*100:.0f}%, Hourly Vol={asset.hourly_vol*100:.3f}%, Per-Second={asset.second_vol*10000:.4f}bps")

=== FEE STRUCTURE v5 ===
Open Fee: 0.00%
Close Fee: max(0.01% notional, 20% profit)
Liquidation: 70% margin lost
Funding: 0.010% per hour

=== ASSETS ===
BTC: Annual Vol=60%, Hourly Vol=0.641%, Per-Second=1.0684bps
ETH: Annual Vol=75%, Hourly Vol=0.801%, Per-Second=1.3355bps
SOL: Annual Vol=100%, Hourly Vol=1.068%, Per-Second=1.7807bps


## Cell 2: Liquidation Buffer Analysis at 70%

In [14]:
def analyze_liquidation_buffer(leverage: int, liq_threshold: float):
    """Calculate liquidation buffer in bps of price move"""
    # At liquidation, margin lost = liq_threshold
    # margin_lost = leverage * price_move
    # liq_threshold = leverage * price_move
    # price_move = liq_threshold / leverage
    
    liq_price_move = liq_threshold / leverage
    liq_buffer_bps = liq_price_move * 10000
    
    return liq_buffer_bps

print("=== LIQUIDATION BUFFER COMPARISON ===")
print(f"{'Leverage':<10} {'70% Liq Buffer (bps)':<20}")
print("-" * 35)

for leverage in [100, 200, 500, 1000, 2000]:
    buf_70 = analyze_liquidation_buffer(leverage, 0.70)
    print(f"{leverage}x{'':<6} {buf_70:<20.2f}")

print()
print("At 1000x with 70% threshold:")
print(f"  - Buffer: {analyze_liquidation_buffer(1000, 0.70):.2f} bps")
print(f"  - User loses 70% of margin before liquidation")

=== LIQUIDATION BUFFER COMPARISON ===
Leverage   70% Liq Buffer (bps)
-----------------------------------
100x       70.00               
200x       35.00               
500x       14.00               
1000x       7.00                
2000x       3.50                

At 1000x with 70% threshold:
  - Buffer: 7.00 bps
  - User loses 70% of margin before liquidation


## Cell 3: Perfect Oracle Scenario - Monte Carlo Simulation

In [4]:
def simulate_trade_perfect_oracle(
    asset: Asset,
    leverage: int,
    margin: float,
    is_long: bool,
    hold_seconds: int,
    fees: FeeStructure
) -> Dict:
    """
    Simulate a single trade with perfect oracle (no front-running possible).
    Returns trade outcome from POOL's perspective.
    """
    notional = margin * leverage
    
    # Simulate price path
    dt = 1  # 1 second steps
    steps = hold_seconds
    
    # Generate price moves
    returns = np.random.normal(0, asset.second_vol, steps)
    cumulative_return = np.cumsum(returns)
    
    # Direction multiplier
    direction = 1 if is_long else -1
    
    # Track PnL at each step
    pnl_path = direction * cumulative_return * notional
    margin_remaining = margin + pnl_path
    
    # Check for liquidation
    liq_threshold_amount = margin * fees.liquidation_threshold
    liquidated_mask = pnl_path <= -liq_threshold_amount
    
    if liquidated_mask.any():
        liq_step = np.argmax(liquidated_mask)
        final_pnl = -liq_threshold_amount  # Trader loses this much
        outcome = 'liquidated'
        # Pool captures remaining margin after liquidation
        pool_pnl = margin  # Pool takes entire margin
        fee_collected = 0  # No close fee on liquidation
    else:
        final_pnl = pnl_path[-1]
        outcome = 'closed'
        
        # Calculate close fee
        min_fee = fees.close_fee_min * notional
        profit_fee = max(0, final_pnl) * fees.profit_fee_rate
        fee_collected = max(min_fee, profit_fee)
        
        # Pool PnL = -trader_pnl + fees
        pool_pnl = -final_pnl + fee_collected
    
    return {
        'asset': asset.name,
        'leverage': leverage,
        'margin': margin,
        'notional': notional,
        'outcome': outcome,
        'trader_pnl': final_pnl,
        'pool_pnl': pool_pnl,
        'fee_collected': fee_collected,
        'hold_seconds': hold_seconds
    }

def run_monte_carlo_perfect(
    asset: Asset,
    leverage: int,
    margin: float,
    n_simulations: int,
    hold_seconds: int,
    fees: FeeStructure
) -> pd.DataFrame:
    """Run Monte Carlo simulation for perfect oracle scenario"""
    results = []
    for _ in range(n_simulations):
        is_long = np.random.random() > 0.5
        result = simulate_trade_perfect_oracle(asset, leverage, margin, is_long, hold_seconds, fees)
        results.append(result)
    return pd.DataFrame(results)

# Run simulation for each asset at 1000x
print("=== PERFECT ORACLE: MONTE CARLO (100K trades per asset) ===")
print(f"Parameters: 1000x leverage, $10 margin, 120s hold, 70% liq threshold")
print()

perfect_results = {}
for asset_name, asset in ASSETS.items():
    df = run_monte_carlo_perfect(asset, 1000, 10, 100000, 120, FEES)
    perfect_results[asset_name] = df
    
    liq_rate = (df['outcome'] == 'liquidated').mean() * 100
    avg_pool_pnl = df['pool_pnl'].mean()
    avg_fee = df[df['outcome'] == 'closed']['fee_collected'].mean()
    total_pool_pnl = df['pool_pnl'].sum()
    
    print(f"{asset_name}:")
    print(f"  Liquidation Rate: {liq_rate:.1f}%")
    print(f"  Avg Pool PnL/trade: ${avg_pool_pnl:.2f}")
    print(f"  Avg Close Fee: ${avg_fee:.3f}")
    print(f"  Total Pool PnL (100K trades): ${total_pool_pnl:,.0f}")
    print()

=== PERFECT ORACLE: MONTE CARLO (100K trades per asset) ===
Parameters: 1000x leverage, $10 margin, 120s hold, 70% liq threshold

BTC:
  Liquidation Rate: 51.4%
  Avg Pool PnL/trade: $2.20
  Avg Close Fee: $1.977
  Total Pool PnL (100K trades): $219,610

ETH:
  Liquidation Rate: 59.4%
  Avg Pool PnL/trade: $2.34
  Avg Close Fee: $2.551
  Total Pool PnL (100K trades): $233,848

SOL:
  Liquidation Rate: 68.1%
  Avg Pool PnL/trade: $2.47
  Avg Close Fee: $3.608
  Total Pool PnL (100K trades): $247,336



## Cell 4: Front-runnable Oracle - Arbitrage Analysis

In [5]:
def simulate_oracle_arbitrage(
    asset: Asset,
    leverage: int,
    margin: float,
    oracle_latency_ms: int,
    fees: FeeStructure,
    n_attempts: int
) -> Dict:
    """
    Simulate oracle front-running attack.
    Attacker sees price move, opens position on stale oracle, closes for profit.
    """
    latency_seconds = oracle_latency_ms / 1000
    notional = margin * leverage
    
    # Threshold: attacker needs price move > close_fee to profit
    min_profitable_move = fees.close_fee_min  # 0.01% = 1 bip
    
    results = []
    for _ in range(n_attempts):
        # Price move during oracle latency window (correct volatility scaling)
        # second_vol * sqrt(latency_seconds) gives volatility over the window
        price_move = np.random.normal(0, asset.second_vol * np.sqrt(latency_seconds))
        price_move_bps = abs(price_move) * 10000
        
        # Attacker captures the move (goes long if price up, short if down)
        gross_profit = abs(price_move) * notional
        
        # Pay close fee (min fee since this is quick arb)
        close_fee = fees.close_fee_min * notional
        
        net_profit = gross_profit - close_fee
        is_profitable = net_profit > 0
        
        results.append({
            'price_move_bps': price_move_bps,
            'gross_profit': gross_profit,
            'close_fee': close_fee,
            'net_profit': net_profit,
            'is_profitable': is_profitable
        })
    
    df = pd.DataFrame(results)
    return {
        'success_rate': df['is_profitable'].mean() * 100,
        'avg_net_profit': df['net_profit'].mean(),
        'total_net_profit': df['net_profit'].sum(),
        'avg_price_move_bps': df['price_move_bps'].mean(),
        'profitable_attempts': df['is_profitable'].sum(),
        'df': df
    }

print("=== FRONT-RUNNABLE ORACLE: ARBITRAGE ANALYSIS ===")
print(f"Oracle latency: 400ms (Pyth)")
print(f"Close fee: {FEES.close_fee_min*100:.2f}% = {FEES.close_fee_min*10000:.0f} bps")
print(f"100,000 arb attempts per asset")
print()

arb_results = {}
for asset_name, asset in ASSETS.items():
    result = simulate_oracle_arbitrage(asset, 1000, 10, 400, FEES, 100000)
    arb_results[asset_name] = result
    
    print(f"{asset_name} (Vol: {asset.annual_vol*100:.0f}%/yr):")
    print(f"  Avg price move in 400ms: {result['avg_price_move_bps']:.2f} bps")
    print(f"  Arb Success Rate: {result['success_rate']:.1f}%")
    print(f"  Avg Net Profit/attempt: ${result['avg_net_profit']:.4f}")
    print(f"  Total Profit (100K attempts): ${result['total_net_profit']:,.2f}")
    status = "VULNERABLE" if result['avg_net_profit'] > 0 else "PROTECTED"
    print(f"  Status: {status}")
    print()

=== FRONT-RUNNABLE ORACLE: ARBITRAGE ANALYSIS ===
Oracle latency: 400ms (Pyth)
Close fee: 0.01% = 1 bps
100,000 arb attempts per asset

BTC (Vol: 60%/yr):
  Avg price move in 400ms: 0.54 bps
  Arb Success Rate: 13.8%
  Avg Net Profit/attempt: $-0.4624
  Total Profit (100K attempts): $-46,235.94
  Status: PROTECTED

ETH (Vol: 75%/yr):
  Avg price move in 400ms: 0.68 bps
  Arb Success Rate: 23.8%
  Avg Net Profit/attempt: $-0.3229
  Total Profit (100K attempts): $-32,293.50
  Status: PROTECTED

SOL (Vol: 100%/yr):
  Avg price move in 400ms: 0.90 bps
  Arb Success Rate: 37.4%
  Avg Net Profit/attempt: $-0.1012
  Total Profit (100K attempts): $-10,121.20
  Status: PROTECTED



## Cell 5: Close Fee Sensitivity - Finding the Break-even

In [6]:
def test_close_fee_threshold(asset: Asset, close_fee_bps: float, n_attempts: int = 50000) -> Dict:
    """Test if a given close fee protects against oracle arb"""
    fees = FeeStructure()
    fees.close_fee_min = close_fee_bps / 10000
    
    result = simulate_oracle_arbitrage(asset, 1000, 10, 400, fees, n_attempts)
    return {
        'close_fee_bps': close_fee_bps,
        'success_rate': result['success_rate'],
        'avg_net_profit': result['avg_net_profit'],
        'total_profit': result['total_net_profit'],
        'is_protected': result['avg_net_profit'] < 0
    }

print("=== CLOSE FEE SENSITIVITY ANALYSIS ===")
print("Finding minimum close fee to protect against 400ms oracle latency")
print()

fee_tests = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]

for asset_name, asset in ASSETS.items():
    print(f"\n{asset_name} ({asset.annual_vol*100:.0f}% annual vol):")
    print(f"{'Fee (bps)':<12} {'Success %':<12} {'Avg Profit':<15} {'Status':<12}")
    print("-" * 50)
    
    for fee_bps in fee_tests:
        result = test_close_fee_threshold(asset, fee_bps)
        status = "PROTECTED" if result['is_protected'] else "VULNERABLE"
        print(f"{fee_bps:<12.2f} {result['success_rate']:<12.1f} ${result['avg_net_profit']:<14.4f} {status}")

=== CLOSE FEE SENSITIVITY ANALYSIS ===
Finding minimum close fee to protect against 400ms oracle latency


BTC (60% annual vol):
Fee (bps)    Success %    Avg Profit      Status      
--------------------------------------------------
0.50         46.1         $0.0392         VULNERABLE
0.75         26.6         $-0.2113        PROTECTED
1.00         13.9         $-0.4627        PROTECTED
1.25         6.3          $-0.7115        PROTECTED
1.50         2.7          $-0.9577        PROTECTED
1.75         1.0          $-1.2092        PROTECTED
2.00         0.3          $-1.4601        PROTECTED
2.50         0.0          $-1.9619        PROTECTED
3.00         0.0          $-2.4574        PROTECTED

ETH (75% annual vol):
Fee (bps)    Success %    Avg Profit      Status      
--------------------------------------------------
0.50         55.2         $0.1705         VULNERABLE
0.75         37.4         $-0.0764        PROTECTED
1.00         23.6         $-0.3291        PROTECTED
1.25      

## Cell 6: Per-Asset Detailed Assessment

In [9]:
def comprehensive_asset_analysis(
    asset: Asset,
    leverage_tiers: list,
    fees: FeeStructure,
    n_sims: int = 50000
) -> pd.DataFrame:
    """Comprehensive analysis per asset across leverage tiers"""
    results = []
    
    for leverage in leverage_tiers:
        # Perfect oracle simulation
        df = run_monte_carlo_perfect(asset, leverage, 10, n_sims, 120, fees)
        
        liq_rate = (df['outcome'] == 'liquidated').mean()
        closed_trades = df[df['outcome'] == 'closed']
        
        # Winning trades (trader profit > 0)
        winning_trades = closed_trades[closed_trades['trader_pnl'] > 0]
        losing_trades = closed_trades[closed_trades['trader_pnl'] <= 0]
        
        results.append({
            'Asset': asset.name,
            'Leverage': f"{leverage}x",
            'Liq Rate': f"{liq_rate*100:.1f}%",
            'Close Rate': f"{(1-liq_rate)*100:.1f}%",
            'Win Rate': f"{len(winning_trades)/len(df)*100:.1f}%",
            'Avg Pool PnL': f"${df['pool_pnl'].mean():.2f}",
            'Pool EV/trade': f"${df['pool_pnl'].mean():.3f}",
            'Liq Buffer (bps)': f"{fees.liquidation_threshold/leverage*10000:.1f}"
        })
    
    return pd.DataFrame(results)

print("=== PER-ASSET COMPREHENSIVE ASSESSMENT ===")
print(f"Close Fee: 1 bps | Liquidation: 70%")
print()

leverage_tiers = [100, 200, 500, 750, 1000, 1500, 2000]

all_assessments = []
for asset_name, asset in ASSETS.items():
    df = comprehensive_asset_analysis(asset, leverage_tiers, FEES)
    all_assessments.append(df)
    print(f"\n{asset_name}:")
    print(df.to_string(index=False))

combined = pd.concat(all_assessments, ignore_index=True)
print("\n\n=== COMBINED ASSESSMENT ===")
print(combined.to_string(index=False))

=== PER-ASSET COMPREHENSIVE ASSESSMENT ===
Close Fee: 1 bps | Liquidation: 70%


BTC:
Asset Leverage Liq Rate Close Rate Win Rate Avg Pool PnL Pool EV/trade Liq Buffer (bps)
  BTC     100x     0.0%     100.0%    50.5%        $0.14        $0.142             70.0
  BTC     200x     0.2%      99.8%    50.0%        $0.31        $0.308             35.0
  BTC     500x    21.3%      78.7%    49.2%        $1.23        $1.232             14.0
  BTC     750x    39.8%      60.2%    45.3%        $1.88        $1.876              9.3
  BTC    1000x    51.2%      48.8%    40.5%        $2.16        $2.156              7.0
  BTC    1500x    65.6%      34.4%    31.3%        $2.60        $2.598              4.7
  BTC    2000x    72.4%      27.6%    26.0%        $2.57        $2.570              3.5

ETH:
Asset Leverage Liq Rate Close Rate Win Rate Avg Pool PnL Pool EV/trade Liq Buffer (bps)
  ETH     100x     0.0%     100.0%    50.1%        $0.17        $0.169             70.0
  ETH     200x     1.4%     

## Cell 7: Liquidation Threshold Verification (70%)

In [11]:
def compare_liquidation_thresholds(
    asset: Asset,
    leverage: int,
    n_sims: int = 50000
) -> Dict:
    """Verify pool profitability at 70% liquidation threshold"""
    
    results = {}
    for threshold in [0.70]:
        fees = FeeStructure(liquidation_threshold=threshold)
        df = run_monte_carlo_perfect(asset, leverage, 10, n_sims, 120, fees)
        
        liq_rate = (df['outcome'] == 'liquidated').mean()
        avg_pool_pnl = df['pool_pnl'].mean()
        
        results[f"{int(threshold*100)}%"] = {
            'liq_rate': liq_rate,
            'avg_pool_pnl': avg_pool_pnl,
            'buffer_bps': threshold / leverage * 10000
        }
    
    return results

print("=== LIQUIDATION THRESHOLD: 70% ===")
print()

for asset_name, asset in ASSETS.items():
    print(f"\n{asset_name}:")
    print(f"{'Leverage':<10} {'Threshold':<12} {'Buffer':<12} {'Liq Rate':<12} {'Pool EV':<12}")
    print("-" * 60)
    
    for leverage in [500, 750, 1000, 1500, 2000]:
        comparison = compare_liquidation_thresholds(asset, leverage)
        for threshold, data in comparison.items():
            print(f"{leverage}x{'':<6} {threshold:<12} {data['buffer_bps']:<12.1f} {data['liq_rate']*100:<12.1f}% ${data['avg_pool_pnl']:<11.2f}")

=== LIQUIDATION THRESHOLD: 70% ===


BTC:
Leverage   Threshold    Buffer       Liq Rate     Pool EV     
------------------------------------------------------------
500x       70%          14.0         21.2        % $1.23       
750x       70%          9.3          39.7        % $1.87       
1000x       70%          7.0          51.4        % $2.19       
1500x       70%          4.7          65.8        % $2.68       
2000x       70%          3.5          72.6        % $2.55       

ETH:
Leverage   Threshold    Buffer       Liq Rate     Pool EV     
------------------------------------------------------------
500x       70%          14.0         31.0        % $1.49       
750x       70%          9.3          48.6        % $2.00       
1000x       70%          7.0          59.5        % $2.33       
1500x       70%          4.7          71.0        % $2.56       
2000x       70%          3.5          77.1        % $2.50       

SOL:
Leverage   Threshold    Buffer       Liq Rate     Po

## Cell 8: Final Recommendation Summary

In [12]:
print("="*70)
print("                    FINAL ANALYSIS SUMMARY                          ")
print("="*70)
print()
print("SCENARIO 1: PERFECT ORACLE (No front-running)")
print("-" * 50)
print("  - 1 bps close fee is VIABLE")
print("  - Pool maintains positive EV across all assets")
print("  - 70% liquidation threshold works well")
print()

print("SCENARIO 2: FRONT-RUNNABLE ORACLE (400ms Pyth latency)")
print("-" * 50)

# Summarize arb vulnerability
for asset_name, result in arb_results.items():
    status = "VULNERABLE" if result['avg_net_profit'] > 0 else "PROTECTED"
    print(f"  {asset_name}: {status} (Arb success: {result['success_rate']:.1f}%, Avg profit: ${result['avg_net_profit']:.4f})")

print()
print("CRITICAL FINDING:")
print("-" * 50)

# Check if any asset is vulnerable
vulnerable = [name for name, r in arb_results.items() if r['avg_net_profit'] > 0]
if vulnerable:
    print(f"  WARNING: {', '.join(vulnerable)} vulnerable to oracle arb at 1 bps!")
    print("  Recommended minimum close fee: 2 bps (0.02%)")
else:
    print("  All assets protected at 1 bps close fee!")

print()
print("PARAMETER RECOMMENDATIONS:")
print("-" * 50)
print("  If using PERFECT oracle (sub-100ms, manipulation-resistant):")
print("    - Close fee: 1 bps (0.01%) is viable")
print("    - Liquidation: 70% threshold (standard)")
print()
print("  If using STANDARD oracle (Pyth 400ms):")
print("    - Close fee: 2 bps (0.02%) minimum recommended")
print("    - Or implement execution delay / hold time")
print("="*70)

                    FINAL ANALYSIS SUMMARY                          

SCENARIO 1: PERFECT ORACLE (No front-running)
--------------------------------------------------
  - 1 bps close fee is VIABLE
  - Pool maintains positive EV across all assets
  - 70% liquidation threshold works well

SCENARIO 2: FRONT-RUNNABLE ORACLE (400ms Pyth latency)
--------------------------------------------------
  BTC: PROTECTED (Arb success: 13.8%, Avg profit: $-0.4624)
  ETH: PROTECTED (Arb success: 23.8%, Avg profit: $-0.3229)
  SOL: PROTECTED (Arb success: 37.4%, Avg profit: $-0.1012)

CRITICAL FINDING:
--------------------------------------------------
  All assets protected at 1 bps close fee!

PARAMETER RECOMMENDATIONS:
--------------------------------------------------
  If using PERFECT oracle (sub-100ms, manipulation-resistant):
    - Close fee: 1 bps (0.01%) is viable
    - Liquidation: 70% threshold (standard)

  If using STANDARD oracle (Pyth 400ms):
    - Close fee: 2 bps (0.02%) minimum recom

## Cell 9: Pool Profitability at 1 BPS - Detailed Breakdown

In [13]:
print("=== POOL PROFITABILITY AT 1 BPS CLOSE FEE ===")
print("Perfect Oracle Scenario")
print()

for asset_name, df in perfect_results.items():
    print(f"\n{asset_name}:")
    
    # Break down by outcome
    liq_trades = df[df['outcome'] == 'liquidated']
    closed_trades = df[df['outcome'] == 'closed']
    winning_closed = closed_trades[closed_trades['trader_pnl'] > 0]
    losing_closed = closed_trades[closed_trades['trader_pnl'] <= 0]
    
    print(f"  Total trades: {len(df):,}")
    print(f"  Liquidations: {len(liq_trades):,} ({len(liq_trades)/len(df)*100:.1f}%)")
    print(f"    Pool profit from liquidations: ${liq_trades['pool_pnl'].sum():,.0f}")
    print(f"  Closed trades: {len(closed_trades):,} ({len(closed_trades)/len(df)*100:.1f}%)")
    print(f"    - Winners: {len(winning_closed):,} ({len(winning_closed)/len(df)*100:.1f}%)")
    print(f"    - Losers: {len(losing_closed):,} ({len(losing_closed)/len(df)*100:.1f}%)")
    print(f"  Total fees collected: ${closed_trades['fee_collected'].sum():,.0f}")
    print(f"  Avg fee per closed trade: ${closed_trades['fee_collected'].mean():.3f}")
    print(f"  Net pool PnL: ${df['pool_pnl'].sum():,.0f}")
    print(f"  Pool EV per trade: ${df['pool_pnl'].mean():.3f}")

=== POOL PROFITABILITY AT 1 BPS CLOSE FEE ===
Perfect Oracle Scenario


BTC:
  Total trades: 100,000
  Liquidations: 51,412 (51.4%)
    Pool profit from liquidations: $514,120
  Closed trades: 48,588 (48.6%)
    - Winners: 40,343 (40.3%)
    - Losers: 8,245 (8.2%)
  Total fees collected: $96,079
  Avg fee per closed trade: $1.977
  Net pool PnL: $219,610
  Pool EV per trade: $2.196

ETH:
  Total trades: 100,000
  Liquidations: 59,448 (59.4%)
    Pool profit from liquidations: $594,480
  Closed trades: 40,552 (40.6%)
    - Winners: 35,764 (35.8%)
    - Losers: 4,788 (4.8%)
  Total fees collected: $103,443
  Avg fee per closed trade: $2.551
  Net pool PnL: $233,848
  Pool EV per trade: $2.338

SOL:
  Total trades: 100,000
  Liquidations: 68,074 (68.1%)
    Pool profit from liquidations: $680,740
  Closed trades: 31,926 (31.9%)
    - Winners: 29,459 (29.5%)
    - Losers: 2,467 (2.5%)
  Total fees collected: $115,179
  Avg fee per closed trade: $3.608
  Net pool PnL: $247,336
  Pool EV per